In [1]:
"""
Classification Loss Functions Survey
======================================

Covers: BCE, CrossEntropy, Focal Loss
When to use which, with concrete comparisons.
"""
import torch
import torch.nn as nn
import matplotlib.pyplot as plt


In [2]:
print("=" * 70)
print("1. BINARY CROSS ENTROPY (BCE)")
print("=" * 70)

print("""
Use when: 2 classes (binary), OR multi-label (each label independent)
Output layer: 1 neuron (binary) or N neurons (multi-label), with Sigmoid
""")

y_true = torch.tensor([1.0])
y_pred = torch.tensor([0.9])  # Model is fairly confident and correct

1. BINARY CROSS ENTROPY (BCE)

Use when: 2 classes (binary), OR multi-label (each label independent)
Output layer: 1 neuron (binary) or N neurons (multi-label), with Sigmoid



In [4]:
bce= nn.BCELoss()
loss= bce(y_pred,y_true)
print(f"True=1, Pred=0.9 (confident+correct) → Loss = {loss.item():.4f}")

True=1, Pred=0.9 (confident+correct) → Loss = 0.1054


In [5]:
y_pred_wrong= torch.tensor([0.1])
loss_wrong= bce(y_pred_wrong,y_true)
print(f"True=1, Pred=0.1 (confident+wrong)   → Loss = {loss_wrong.item():.4f}")

True=1, Pred=0.1 (confident+wrong)   → Loss = 2.3026


In [6]:
print("\n" + "=" * 70)
print("2. CROSSENTROPY LOSS")
print("=" * 70)

print("""
Use when: Multi-class, ONE correct answer (e.g. MNIST: digit is 3, not 3 AND 7)
Output layer: N neurons, NO activation (CrossEntropyLoss applies softmax internally)
""")


2. CROSSENTROPY LOSS

Use when: Multi-class, ONE correct answer (e.g. MNIST: digit is 3, not 3 AND 7)
Output layer: N neurons, NO activation (CrossEntropyLoss applies softmax internally)



In [7]:
outputs = torch.tensor([[2.0, 0.5, 0.3, 5.0, 0.1, 0.2, 0.1, 0.3, 0.2, 0.1]])  # raw logits
label = torch.tensor([3])  # true class is index 3

In [11]:
ce= nn.CrossEntropyLoss()
loss_ce= ce(outputs,label)
print(f"Logits: {outputs.tolist()[0]}")
print(f"True class: {label.item()}")
print(f"Loss: {loss_ce.item():.4f}  (low, because model's highest score IS at index 3)")


Logits: [2.0, 0.5, 0.30000001192092896, 5.0, 0.10000000149011612, 0.20000000298023224, 0.10000000149011612, 0.30000001192092896, 0.20000000298023224, 0.10000000149011612]
True class: 3
Loss: 0.1114  (low, because model's highest score IS at index 3)


In [12]:
# Now wrong prediction
outputs_wrong = torch.tensor([[5.0, 0.5, 0.3, 0.2, 0.1, 0.2, 0.1, 0.3, 0.2, 0.1]])
loss_wrong = ce(outputs_wrong, label)
print(f"\nSame true class, but highest score is index 0 (wrong)")
print(f"Loss: {loss_wrong.item():.4f}  (much higher)")


Same true class, but highest score is index 0 (wrong)
Loss: 4.8736  (much higher)


In [13]:
print("""
COMMON MISTAKE:
  Using BCELoss when you actually have a multi-class problem.
  BCELoss expects outputs.shape == labels.shape (both one-hot-like)
  CrossEntropyLoss expects outputs=(batch, classes), labels=(batch,) class indices
""")


COMMON MISTAKE:
  Using BCELoss when you actually have a multi-class problem.
  BCELoss expects outputs.shape == labels.shape (both one-hot-like)
  CrossEntropyLoss expects outputs=(batch, classes), labels=(batch,) class indices



In [14]:
print("\n" + "=" * 70)
print("3. FOCAL LOSS")
print("=" * 70)

print("""
Use when: SEVERE class imbalance (rare class getting ignored by CrossEntropy)
Common in: object detection (mostly background pixels, few actual objects),
           rare character recognition (your Devanagari conjuncts later!)

Formula: FL(p_t) = -(1 - p_t)^gamma * log(p_t)

Where p_t = model's predicted probability for the TRUE class
      gamma = focusing parameter (typically 2)

Key idea: when p_t is HIGH (model already confident and correct),
          (1-p_t)^gamma becomes very small → loss contribution shrinks
          
          when p_t is LOW (model wrong or unsure),
          (1-p_t)^gamma stays close to 1 → loss barely reduced

This means: EASY, already-correct examples contribute almost nothing to
            the loss. HARD, wrong examples dominate the gradient signal.
            Standard CrossEntropy treats both equally.
""")



3. FOCAL LOSS

Use when: SEVERE class imbalance (rare class getting ignored by CrossEntropy)
Common in: object detection (mostly background pixels, few actual objects),
           rare character recognition (your Devanagari conjuncts later!)

Formula: FL(p_t) = -(1 - p_t)^gamma * log(p_t)

Where p_t = model's predicted probability for the TRUE class
      gamma = focusing parameter (typically 2)

Key idea: when p_t is HIGH (model already confident and correct),
          (1-p_t)^gamma becomes very small → loss contribution shrinks
          
          when p_t is LOW (model wrong or unsure),
          (1-p_t)^gamma stays close to 1 → loss barely reduced

This means: EASY, already-correct examples contribute almost nothing to
            the loss. HARD, wrong examples dominate the gradient signal.
            Standard CrossEntropy treats both equally.



In [15]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, outputs, targets):
        ce_loss = nn.functional.cross_entropy(outputs, targets, reduction='none')
        p_t = torch.exp(-ce_loss)  # recover p_t from CE loss
        focal_loss = ((1 - p_t) ** self.gamma) * ce_loss
        return focal_loss.mean()

In [16]:
# Compare CrossEntropy vs Focal on an EASY (already correct) example
easy_output = torch.tensor([[5.0, 0.1, 0.1]])
easy_label = torch.tensor([0])

ce_easy = nn.functional.cross_entropy(easy_output, easy_label)
focal = FocalLoss(gamma=2.0)
focal_easy = focal(easy_output, easy_label)

print(f"EASY example (model already confident+correct):")
print(f"  CrossEntropy loss: {ce_easy.item():.4f}")
print(f"  Focal loss:        {focal_easy.item():.4f}  (much smaller!)")

EASY example (model already confident+correct):
  CrossEntropy loss: 0.0148
  Focal loss:        0.0000  (much smaller!)


In [17]:
# Compare on a HARD (wrong) example
hard_output = torch.tensor([[0.1, 0.1, 5.0]])
hard_label = torch.tensor([0])

ce_hard = nn.functional.cross_entropy(hard_output, hard_label)
focal_hard = focal(hard_output, hard_label)

print(f"\nHARD example (model wrong, confident about wrong class):")
print(f"  CrossEntropy loss: {ce_hard.item():.4f}")
print(f"  Focal loss:        {focal_hard.item():.4f}  (similar magnitude, not shrunk)")

print("""
Result: Focal loss DOWN-WEIGHTS easy examples, letting hard/rare 
examples dominate training. This is why it's used for imbalanced 
detection/classification — forces the model to focus on what it's 
actually struggling with.
""")


HARD example (model wrong, confident about wrong class):
  CrossEntropy loss: 4.9148
  Focal loss:        4.8429  (similar magnitude, not shrunk)

Result: Focal loss DOWN-WEIGHTS easy examples, letting hard/rare 
examples dominate training. This is why it's used for imbalanced 
detection/classification — forces the model to focus on what it's 
actually struggling with.

